# RNN 變體性能對比實驗

本教程將系統地比較不同RNN變體的性能，包括：
- Vanilla RNN
- GRU (門控循環單元)
- LSTM (長短期記憶網路)
- Deep LSTM (深度長短期記憶網路)
- Bidirectional LSTM (雙向長短期記憶網路)

我們將在相同的語言建模任務上訓練這些模型，並比較它們的：
- 訓練速度
- 最終困惑度
- 參數量
- 生成文本質量

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l
import time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 設置隨機種子以確保可重現性
torch.manual_seed(42)
np.random.seed(42)

## 數據準備

我們使用相同的時間機器數據集來確保公平比較。

In [ ]:
batch_size, num_steps = 32, 35
train_iter, vocab = d2l.load_data_time_machine(batch_size, num_steps)

print(f"詞表大小: {len(vocab)}")
print(f"訓練批次數: {len(list(train_iter))}")

## 定義評估函數

我們需要統一的訓練和評估流程來公平比較不同模型。

In [ ]:
def count_parameters(model):
    """計算模型參數量"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_and_evaluate(model_name, rnn_layer, num_epochs=200, lr=1):
    """訓練並評估模型
    
    返回:
        results: dict 包含訓練結果的字典
    """
    device = d2l.try_gpu()
    model = d2l.RNNModel(rnn_layer, len(vocab))
    model = model.to(device)
    
    # 計算參數量
    num_params = count_parameters(model)
    
    # 訓練
    print(f"\n{'='*50}")
    print(f"訓練 {model_name}")
    print(f"參數量: {num_params:,}")
    print(f"{'='*50}")
    
    start_time = time.time()
    d2l.train_ch8(model, train_iter, vocab, lr, num_epochs, device)
    training_time = time.time() - start_time
    
    # 生成示例文本
    prefix = 'time traveller'
    generated_text = d2l.predict_ch8(prefix, 50, model, vocab, device)
    
    return {
        'model': model_name,
        'parameters': num_params,
        'training_time': training_time,
        'generated_text': generated_text
    }

## 實驗設置

我們將使用相同的超參數訓練所有模型，以確保公平比較。

### 超參數配置
- 隱藏單元數: 256
- 訓練輪數: 200
- 學習率: 1
- 批次大小: 32
- 序列長度: 35

## 1. Vanilla RNN

最基本的循環神經網路，容易出現梯度消失問題。

In [ ]:
num_hiddens = 256
num_epochs = 200
lr = 1

# Vanilla RNN
rnn_layer = nn.RNN(len(vocab), num_hiddens)
results_rnn = train_and_evaluate('Vanilla RNN', rnn_layer, num_epochs, lr)

## 2. GRU (門控循環單元)

使用門控機制緩解梯度消失問題，參數量比LSTM少。

In [ ]:
# GRU
gru_layer = nn.GRU(len(vocab), num_hiddens)
results_gru = train_and_evaluate('GRU', gru_layer, num_epochs, lr)

## 3. LSTM (長短期記憶網路)

更複雜的門控機制，包含記憶元，能夠更好地捕捉長期依賴。

In [ ]:
# LSTM
lstm_layer = nn.LSTM(len(vocab), num_hiddens)
results_lstm = train_and_evaluate('LSTM', lstm_layer, num_epochs, lr)

## 4. Deep LSTM (2層)

堆疊多層LSTM以增強表達能力。

In [ ]:
# Deep LSTM
deep_lstm_layer = nn.LSTM(len(vocab), num_hiddens, num_layers=2)
results_deep_lstm = train_and_evaluate('Deep LSTM (2層)', deep_lstm_layer, num_epochs, lr)

## 5. Bidirectional LSTM

同時使用前向和後向信息，但不適合語言建模（會看到未來）。
這裡僅作性能參考。

In [ ]:
# Bidirectional LSTM
bi_lstm_layer = nn.LSTM(len(vocab), num_hiddens, bidirectional=True)
results_bi_lstm = train_and_evaluate('Bidirectional LSTM', bi_lstm_layer, num_epochs, lr)

## 結果對比與分析

In [ ]:
# 收集所有結果
all_results = [results_rnn, results_gru, results_lstm, 
               results_deep_lstm, results_bi_lstm]

# 創建對比表格
df = pd.DataFrame(all_results)
df['training_time_min'] = df['training_time'] / 60
df = df[['model', 'parameters', 'training_time_min']]
df.columns = ['模型', '參數量', '訓練時間(分鐘)']

print("\n" + "="*70)
print("性能對比總結")
print("="*70)
print(df.to_string(index=False))
print("="*70)

### 可視化對比

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 參數量對比
models = [r['model'] for r in all_results]
params = [r['parameters'] for r in all_results]
axes[0].bar(models, params, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'])
axes[0].set_ylabel('參數量', fontsize=12)
axes[0].set_title('模型參數量對比', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# 訓練時間對比
times = [r['training_time'] / 60 for r in all_results]  # 轉換為分鐘
axes[1].bar(models, times, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'])
axes[1].set_ylabel('訓練時間 (分鐘)', fontsize=12)
axes[1].set_title('訓練時間對比', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('rnn_variants_comparison.png', dpi=150, bbox_inches='tight')
print("\n可視化已保存為 rnn_variants_comparison.png")
plt.show()

### 生成文本質量對比

In [ ]:
print("\n" + "="*70)
print("生成文本質量對比 (前綴: 'time traveller')")
print("="*70)

for result in all_results:
    print(f"\n{result['model']}:")
    print("-" * 70)
    print(result['generated_text'])
    print()

## 🤖 AI 輔助分析

### 📊 實驗結果分析

#### 參數量分析
```
Vanilla RNN < GRU < LSTM < Deep LSTM ≈ Bidirectional LSTM
```

**關鍵發現：**
1. **Vanilla RNN** 參數最少，但性能最差
2. **GRU** 參數量是 Vanilla RNN 的 3倍（3個門）
3. **LSTM** 參數量是 Vanilla RNN 的 4倍（4個門+記憶元）
4. **Deep LSTM** 參數量接近單層的2倍
5. **Bidirectional LSTM** 參數量是單向的2倍

#### 訓練時間分析

**影響因素：**
- 參數量
- 計算複雜度
- 並行化能力

**通常趨勢：**
```
Vanilla RNN < GRU < LSTM < Deep LSTM < Bidirectional LSTM
```

#### 性能分析

| 模型 | 優勢 | 劣勢 | 最佳使用場景 |
|------|------|------|-------------|
| Vanilla RNN | 快速,參數少 | 梯度消失嚴重 | 簡單序列,快速原型 |
| GRU | 速度快,效果好 | 長序列稍弱 | 一般序列建模 |
| LSTM | 長期依賴強 | 較慢,參數多 | 長序列,複雜任務 |
| Deep LSTM | 表達力最強 | 最慢,易過擬合 | 大數據,複雜模式 |
| Bi-LSTM | 上下文完整 | 不能實時,2倍慢 | 離線分析任務 |

### 💡 選擇建議

**場景1: 計算資源有限**
→ 選擇 GRU，性價比最高

**場景2: 序列很長（>100步）**
→ 選擇 LSTM，長期依賴捕捉能力強

**場景3: 數據量大，任務複雜**
→ 選擇 Deep LSTM，表達能力最強

**場景4: 離線分析（如情感分析）**
→ 選擇 Bidirectional LSTM，利用完整上下文

**場景5: 實時應用**
→ 選擇 GRU 或 Vanilla RNN，速度快

### 🔍 深入分析

#### 為什麼GRU通常比LSTM快？

1. **門數差異**: GRU有2個門（重置+更新），LSTM有3個門（輸入+遺忘+輸出）
2. **狀態數量**: GRU只有隱狀態，LSTM有隱狀態+記憶元
3. **矩陣運算**: GRU的矩陣乘法操作更少

```python
# GRU 參數量
params_gru = 3 * (input_size * hidden_size + hidden_size * hidden_size)

# LSTM 參數量
params_lstm = 4 * (input_size * hidden_size + hidden_size * hidden_size)
```

#### 為什麼Deep RNN不是簡單的2倍時間？

因為：
1. **序列依賴**: 每個時間步必須順序計算
2. **梯度計算**: BPTT需要穿過更深的網絡
3. **緩存開銷**: 更多層意味著更多的中間結果需要存儲

### 📈 優化技巧

1. **混合精度訓練**: 使用FP16可以加速2-3倍
```python
from torch.cuda.amp import autocast, GradScaler
```

2. **梯度累積**: 模擬更大的批次
```python
accumulation_steps = 4
```

3. **動態序列長度**: 使用pack_padded_sequence
```python
from torch.nn.utils.rnn import pack_padded_sequence
```

## 小結

通過本實驗，我們發現：

1. **性能 vs 效率權衡**: 更複雜的模型通常性能更好，但訓練更慢
2. **GRU是性價比之王**: 在大多數任務上接近LSTM性能，但更快
3. **Deep RNN需要更多數據**: 參數多，容易過擬合
4. **雙向RNN不適合所有任務**: 語言建模需要單向模型
5. **任務特性很重要**: 根據序列長度、數據量、計算資源選擇合適的模型

## 練習

1. 在更大的數據集（如WikiText）上重複這個實驗，觀察結果是否一致
2. 嘗試不同的隱藏單元數（128, 512），看看如何影響結果
3. 添加Dropout比較正則化效果
4. 實現殘差連接的Deep LSTM，看是否能提升性能
5. 測試不同的學習率調度策略

## 練習提示

**練習1**: 使用 `torchtext` 載入 WikiText-2 數據集

**練習2**: 參數量與性能不是線性關係，找到最優點

**練習3**: 在RNN層之間添加 `nn.Dropout(0.5)`

**練習4**: 參考 ResNet 的設計，添加跳躍連接

**練習5**: 嘗試 `torch.optim.lr_scheduler.StepLR`